# 04 — Conditional fOU Convergence Signal
Runs only on the final 40 eligible pairs. DTE is the smallest T with conditional convergence probability >=70%.

In [1]:

import pickle
import numpy as np
import pandas as pd

from src.convergence_signal import calculate_convergence_signal, trading_days_to_calendar_days

In [2]:
eligible_pairs = pd.read_parquet("data/processed/eligible_pairs.parquet")
with open("data/processed/spreads.pkl", "rb") as f:
    spreads = pickle.load(f)
print(f"Eligible pairs: {len(eligible_pairs)}")

Eligible pairs: 40


In [3]:
TARGET_PROBABILITY=0.70
MEMORY_WINDOW=60
MAX_HORIZON_DAYS=126
N_PATHS=5000  # switch to 5000 for final run
ENTRY_Z = 1.5

In [4]:
row=eligible_pairs.iloc[0]
history=spreads[(row["dependent"],row["independent"])]
signal, probability_curve = calculate_convergence_signal(history,row["mu"],row["kappa"],row["sigma"],row["hurst"],row["variance"],TARGET_PROBABILITY,ENTRY_Z,MEMORY_WINDOW,MAX_HORIZON_DAYS,N_PATHS,seed=42)
signal

ConvergenceSignal(current_spread=-0.04849624607223557, current_z=-0.9701777262575331, direction=-1, target_probability=0.7, selected_dte_trading_days=34, probability_at_selected_dte=0.7072, probability_at_max_horizon=0.9866, statistical_signal=False)

In [7]:
rows=[]
for _, row in eligible_pairs.iterrows():
    key=(row["dependent"],row["independent"])
    if key not in spreads: continue
    try:
        sig,_=calculate_convergence_signal(spreads[key],row["mu"],row["kappa"],row["sigma"],row["hurst"],row["variance"],TARGET_PROBABILITY,ENTRY_Z,MEMORY_WINDOW,MAX_HORIZON_DAYS,N_PATHS,seed=42)
        rec={"pair":row["pair"],"dependent":row["dependent"],"independent":row["independent"],**sig.to_dict()}
        rec["selected_dte_calendar_days"] = trading_days_to_calendar_days(sig.selected_dte_trading_days) if sig.selected_dte_trading_days is not None else np.nan
        rows.append(rec)
    except Exception as exc:
        print(f"Skipped {row['pair']}: {exc}")
current_signals=pd.DataFrame(rows)
current_signals

,pair,dependent,independent,current_spread,current_z,direction,target_probability,selected_dte_trading_days,probability_at_selected_dte,probability_at_max_horizon,statistical_signal,selected_dte_calendar_days
0,MLM-VMC,MLM,VMC,-0.048496,-0.970178,-1,0.7,34,0.7072,0.9866,False,50
1,SHW-HD,SHW,HD,-0.129302,-2.280265,-1,0.7,61,0.7002,0.9538,True,89
2,URI-MS,URI,MS,0.118563,1.352716,1,0.7,48,0.7022,0.9650,False,70
3,SPGI-MCO,SPGI,MCO,0.010185,0.254589,1,0.7,12,0.7024,0.9862,False,18
4,ORCL-ETN,ORCL,ETN,-0.010272,-0.156772,-1,0.7,10,0.7110,0.9894,False,15
5,HLT-AXP,HLT,AXP,0.023167,0.338066,1,0.7,16,0.7016,0.9840,False,24
6,V-ROP,V,ROP,-0.034033,-0.545488,-1,0.7,29,0.7006,0.9700,False,43
7,PHM-NVR,PHM,NVR,0.007912,0.105622,1,0.7,7,0.7226,0.9878,False,11
8,PEP-HSY,PEP,HSY,0.005337,0.129137,1,0.7,6,0.7002,0.9898,False,9
9,ETN-KLAC,ETN,KLAC,0.052737,0.690021,1,0.7,39,0.7038,0.9552,False,57


In [8]:
candidates=current_signals[current_signals["statistical_signal"]].sort_values(["selected_dte_trading_days","probability_at_max_horizon"],ascending=[True,False]).reset_index(drop=True)
candidates

,pair,dependent,independent,current_spread,current_z,direction,target_probability,selected_dte_trading_days,probability_at_selected_dte,probability_at_max_horizon,statistical_signal,selected_dte_calendar_days
0,SHW-HD,SHW,HD,-0.129302,-2.280265,-1,0.7,61,0.7002,0.9538,True,89
1,WMT-SPGI,WMT,SPGI,0.125245,1.711591,1,0.7,83,0.7034,0.8694,True,121
2,SHW-DHI,SHW,DHI,-0.175236,-1.951851,-1,0.7,84,0.7058,0.8700,True,122
3,RSG-AJG,RSG,AJG,-0.083368,-1.696617,-1,0.7,86,0.7010,0.8548,True,125
4,MAS-LEN,MAS,LEN,-0.197070,-2.414675,-1,0.7,93,0.7010,0.8426,True,135
5,PNR-NWSA,PNR,NWSA,-0.147598,-1.599998,-1,0.7,95,0.7044,0.8212,True,138
6,LYV-AXP,LYV,AXP,-0.204571,-2.174888,-1,0.7,96,0.7056,0.8334,True,140
7,MCO-MSCI,MCO,MSCI,-0.119591,-1.941628,-1,0.7,97,0.7006,0.8176,True,141
8,EMR-TEL,EMR,TEL,0.211552,3.241486,1,0.7,122,0.7032,0.7248,True,177


In [9]:
OUTPUT="data/processed/fou_current_signals.parquet"
current_signals.to_parquet(OUTPUT,index=False)
print(f"Saved: {OUTPUT}")

Saved: data/processed/fou_current_signals.parquet
